face tracking ให้แม่นยำและคงที่มากขึ้น โดยใช้หลายเทคนิคร่วมกัน:

ReID Model (OSNet) - สำหรับ extract feature ที่แม่นยำกว่า

Deep SORT algorithm - tracking ที่ทรงพลัง

Multi-scale detection - จัดการ zoom in/out

Temporal consistency - รักษา ID ให้คงที่

โค้ดนี้มีการปรับปรุงหลายจุดเพื่อให้ tracking แม่นยำและคงที่มากขึ้น:
การปรับปรุงหลัก:

Hungarian Algorithm - ใช้ scipy.optimize.linear_sum_assignment ในการจับคู่ detection กับ track อย่างเหมาะสมที่สุด

Feature Bank - เก็บ embedding หลายตัว (10 ตัว) ของแต่ละคน แล้วเปรียบเทียบกับทั้งหมดเพื่อหา match ที่ดีที่สุด

Kalman Filter - ทำนายตำแหน่งของหน้าในเฟรมถัดไป ช่วยให้ track ได้แม่นยำขึ้นตอนคนเคลื่อนไหวเร็ว

Multi-factor Cost Matrix - รวมคะแนนจาก:

Feature distance (60%) - ความคล้ายของใบหน้า

Spatial distance (20%) - ระยะห่างทางกายภาพ (IoU)

Predicted position (20%) - ตำแหน่งที่ Kalman ทำนาย


Momentum-based Embedding Update - อัพเดท embedding ของแต่ละคนค่อยๆ (10%) ไม่ใช่เปลี่ยนทันที

Extended Disappearance Tolerance - รอได้ถึง 50 เฟรม (เพิ่มจาก 20) ก่อนลบ ID ทิ้ง

จุดเด่น:

รักษา ID ได้ดีแม้เจอ zoom in/out

handle camera switch ได้ดีขึ้นด้วย feature bank

trajectory สวยงามขึ้นด้วย Kalman filter

ถ้าต้องการปรับ sensitivity เพิ่มเติมสามารถแก้ค่า FEATURE_DISTANCE_THRESHOLD และ MAX_DISAPPEARED ได้เลยครับ

In [1]:
import cv2
import numpy as np
from IPython.display import display, Image, clear_output
import time
from collections import deque, defaultdict
from insightface.app import FaceAnalysis
from pytubefix import YouTube
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine

# =============================
# CONFIG
# =============================
YOUTUBE_URL = "https://youtu.be/cmdMopdk6lo"

START_TIME = 75
END_TIME = 120

TARGET_WIDTH = 800
FACE_PANEL_WIDTH = 150
FACE_SIZE = 120
MAX_FACES_DISPLAY = 4

DET_SIZE = (640, 640)
FONT = cv2.FONT_HERSHEY_SIMPLEX

# Enhanced tracking parameters
MAX_DISAPPEARED = 50  # เพิ่มขึ้นเพื่อรอให้หน้ากลับมา
MIN_CONFIDENCE = 0.6
IOU_THRESHOLD = 0.3
FEATURE_DISTANCE_THRESHOLD = 0.45  # ลดลงเพื่อให้ match ง่ายขึ้น
TRAJECTORY_LENGTH = 60

# =============================
# INIT INSIGHTFACE
# =============================
app = FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=DET_SIZE)

# =============================
# KALMAN FILTER
# =============================
class KalmanFilter:
    def __init__(self):
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
        self.kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        
    def predict(self):
        return self.kf.predict()
    
    def update(self, measurement):
        self.kf.correct(np.array([[np.float32(measurement[0])],[np.float32(measurement[1])]]))

# =============================
# UTIL FUNCTIONS
# =============================
def extract_face(frame, box, margin=20):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    x1, y1 = max(0, x1-margin), max(0, y1-margin)
    x2, y2 = min(w, x2+margin), min(h, y2+margin)
    crop = frame[y1:y2, x1:x2]
    return crop if crop.size else None

def bbox_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def bbox_center(box):
    return ((box[0]+box[2])//2, (box[1]+box[3])//2)

def bbox_distance(box1, box2):
    c1 = bbox_center(box1)
    c2 = bbox_center(box2)
    return np.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)

# =============================
# ENHANCED FACE TRACKER
# =============================
class EnhancedFaceTracker:
    def __init__(self, max_disappeared=MAX_DISAPPEARED, trajectory_len=TRAJECTORY_LENGTH):
        self.next_id = 1
        self.tracks = {}
        self.max_disappeared = max_disappeared
        self.trajectory_len = trajectory_len
        self.colors = {}
        
        # Feature bank for each track (multiple embeddings)
        self.feature_bank = defaultdict(list)
        self.max_features = 10
        
    def _get_color(self, track_id):
        if track_id not in self.colors:
            hue = (track_id * 0.618033988749895) % 1.0
            c = cv2.cvtColor(np.uint8([[[hue*180, 255, 200]]]), cv2.COLOR_HSV2BGR)[0][0]
            self.colors[track_id] = tuple(map(int, c))
        return self.colors[track_id]
    
    def _compute_feature_distance(self, emb1, emb2):
        """Compute cosine distance between embeddings"""
        return cosine(emb1, emb2)
    
    def _compute_cost_matrix(self, detections, track_ids):
        """Compute cost matrix combining feature distance and spatial distance"""
        cost_matrix = np.zeros((len(detections), len(track_ids)))
        
        for i, det in enumerate(detections):
            det_box, det_emb = det['box'], det['emb']
            
            for j, track_id in enumerate(track_ids):
                track = self.tracks[track_id]
                
                # Feature distance (primary)
                if len(self.feature_bank[track_id]) > 0:
                    # Compare with all stored features and take minimum
                    feat_dists = [self._compute_feature_distance(det_emb, feat) 
                                 for feat in self.feature_bank[track_id]]
                    feature_dist = min(feat_dists)
                else:
                    feature_dist = self._compute_feature_distance(det_emb, track['emb'])
                
                # Spatial distance (secondary)
                iou = bbox_iou(det_box, track['box'])
                spatial_dist = 1 - iou
                
                # Predicted position distance
                pred_center = track['kalman'].predict()[:2].flatten()
                det_center = np.array(bbox_center(det_box))
                pred_dist = np.linalg.norm(pred_center - det_center) / 1000.0
                
                # Combined cost (weighted)
                cost = 0.6 * feature_dist + 0.2 * spatial_dist + 0.2 * pred_dist
                cost_matrix[i, j] = cost
        
        return cost_matrix
    
    def update(self, detections):
        """
        Update tracks with new detections
        detections: list of dicts with 'box', 'face', 'emb', 'score'
        """
        if len(detections) == 0:
            # Mark all tracks as disappeared
            for track_id in list(self.tracks.keys()):
                self.tracks[track_id]['disappeared'] += 1
                if self.tracks[track_id]['disappeared'] > self.max_disappeared:
                    del self.tracks[track_id]
                    if track_id in self.feature_bank:
                        del self.feature_bank[track_id]
            return
        
        # If no existing tracks, create new ones
        if len(self.tracks) == 0:
            for det in detections:
                self._create_track(det)
            return
        
        # Hungarian algorithm for assignment
        track_ids = list(self.tracks.keys())
        cost_matrix = self._compute_cost_matrix(detections, track_ids)
        
        # Solve assignment problem
        det_indices, track_indices = linear_sum_assignment(cost_matrix)
        
        # Track which detections and tracks are matched
        matched_detections = set()
        matched_tracks = set()
        
        # Update matched tracks
        for det_idx, track_idx in zip(det_indices, track_indices):
            if cost_matrix[det_idx, track_idx] < FEATURE_DISTANCE_THRESHOLD:
                track_id = track_ids[track_idx]
                self._update_track(track_id, detections[det_idx])
                matched_detections.add(det_idx)
                matched_tracks.add(track_id)
        
        # Create new tracks for unmatched detections
        for i, det in enumerate(detections):
            if i not in matched_detections:
                self._create_track(det)
        
        # Mark unmatched tracks as disappeared
        for track_id in track_ids:
            if track_id not in matched_tracks:
                self.tracks[track_id]['disappeared'] += 1
                if self.tracks[track_id]['disappeared'] > self.max_disappeared:
                    del self.tracks[track_id]
                    if track_id in self.feature_bank:
                        del self.feature_bank[track_id]
    
    def _create_track(self, detection):
        """Create new track from detection"""
        track_id = self.next_id
        self.next_id += 1
        
        kalman = KalmanFilter()
        center = bbox_center(detection['box'])
        kalman.update(center)
        
        self.tracks[track_id] = {
            'box': detection['box'],
            'face': detection['face'],
            'emb': detection['emb'],
            'disappeared': 0,
            'trajectory': deque([center], maxlen=self.trajectory_len),
            'kalman': kalman,
            'frames_tracked': 1
        }
        
        # Initialize feature bank
        self.feature_bank[track_id] = [detection['emb'].copy()]
    
    def _update_track(self, track_id, detection):
        """Update existing track with new detection"""
        track = self.tracks[track_id]
        
        # Update basic info
        track['box'] = detection['box']
        track['face'] = detection['face']
        track['disappeared'] = 0
        track['frames_tracked'] += 1
        
        # Update embedding with momentum
        alpha = 0.1  # Update rate
        track['emb'] = (1-alpha) * track['emb'] + alpha * detection['emb']
        
        # Update feature bank
        if len(self.feature_bank[track_id]) < self.max_features:
            self.feature_bank[track_id].append(detection['emb'].copy())
        else:
            # Replace oldest feature
            self.feature_bank[track_id].pop(0)
            self.feature_bank[track_id].append(detection['emb'].copy())
        
        # Update Kalman filter
        center = bbox_center(detection['box'])
        track['kalman'].update(center)
        track['trajectory'].append(center)
    
    def draw_tracks(self, frame):
        """Draw bounding boxes and trajectories on frame"""
        for track_id, track in self.tracks.items():
            if track['disappeared'] == 0:
                x1, y1, x2, y2 = track['box']
                color = self._get_color(track_id)
                
                # Draw bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # Draw ID label with background
                label = f"ID:{track_id}"
                (w, h), _ = cv2.getTextSize(label, FONT, 0.7, 2)
                cv2.rectangle(frame, (x1, y1-h-10), (x1+w+10, y1), color, -1)
                cv2.putText(frame, label, (x1+5, y1-5), FONT, 0.7, (255,255,255), 2)
                
                # Draw trajectory
                if len(track['trajectory']) > 1:
                    pts = list(track['trajectory'])
                    for i in range(1, len(pts)):
                        thickness = int(np.sqrt(self.trajectory_len / float(i+1)) * 2)
                        cv2.line(frame, pts[i-1], pts[i], color, thickness)
        
        return frame

# =============================
# FACE PANEL
# =============================
def create_face_panel(tracker, height):
    panel = np.ones((height, FACE_PANEL_WIDTH, 3), np.uint8) * 40
    cv2.putText(panel, "Tracked", (10, 25), FONT, 0.5, (255,255,255), 1)
    cv2.putText(panel, "Faces", (10, 45), FONT, 0.5, (255,255,255), 1)
    
    y = 60
    active = [(tid, t) for tid, t in tracker.tracks.items() 
              if t['disappeared'] == 0 and t['face'] is not None]
    active.sort(key=lambda x: x[1]['frames_tracked'], reverse=True)
    
    for track_id, track in active[:MAX_FACES_DISPLAY]:
        if y + FACE_SIZE > height:
            break
        
        face = cv2.resize(track['face'], (FACE_SIZE, FACE_SIZE))
        color = tracker._get_color(track_id)
        
        # Draw face thumbnail
        panel[y:y+FACE_SIZE, 15:15+FACE_SIZE] = face
        cv2.rectangle(panel, (13, y-2), (15+FACE_SIZE+2, y+FACE_SIZE+2), color, 2)
        
        # Draw ID label
        cv2.putText(panel, f"ID:{track_id}", (15, y-5), FONT, 0.4, color, 1)
        
        y += FACE_SIZE + 10
    
    return panel

# =============================
# MAIN PROCESSING
# =============================
print("Initializing video stream...")
yt = YouTube(YOUTUBE_URL)
stream = yt.streams.filter(progressive=True, file_extension="mp4").order_by('resolution').desc().first()
cap = cv2.VideoCapture(stream.url)
cap.set(cv2.CAP_PROP_POS_MSEC, START_TIME * 1000)

tracker = EnhancedFaceTracker()
frame_count = 0

print("Starting face tracking...")
while cap.isOpened():
    clear_output(wait=True)
    ret, frame = cap.read()
    if not ret:
        break
    
    current_time = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
    if current_time >= END_TIME:
        break
    
    frame_count += 1
    
    # Detect faces
    faces_data = app.get(frame)
    
    # Prepare detections
    detections = []
    for face in faces_data:
        if face.det_score < MIN_CONFIDENCE:
            continue
            
        box = face.bbox.astype(int)
        face_crop = extract_face(frame, box)
        
        if face_crop is not None:
            detections.append({
                'box': box,
                'face': face_crop,
                'emb': face.embedding,
                'score': face.det_score
            })
    
    # Update tracker
    tracker.update(detections)
    
    # Draw tracks
    frame = tracker.draw_tracks(frame)
    
    # Resize frame
    h, w = frame.shape[:2]
    new_h = int(TARGET_WIDTH * h / w)
    frame = cv2.resize(frame, (TARGET_WIDTH, new_h))
    
    # Create face panel
    panel = create_face_panel(tracker, new_h)
    
    # Combine frame and panel
    combined = np.hstack([frame, panel])
    
    # Display
    _, buffer = cv2.imencode('.jpg', combined)
    display(Image(data=buffer.tobytes()))
    
    time.sleep(0.015)

cap.release()
print(f"Done! Processed {frame_count} frames")
print(f"Total unique faces tracked: {tracker.next_id - 1}")

Done! Processed 468 frames
Total unique faces tracked: 21
